# 🚀 BreathingCNN — Training Notebook (Kaggle GPU)

**Before running:**
1. Settings → Accelerator → **GPU T4 x1** ✅
2. Add Input → attach your dataset `srip-breathing-data` ✅
3. Run All Cells ✅

**What this notebook does:**
```
combined_dataset.csv
  ↓
Cut 15-second windows (50% overlap)
  ↓
Each window → shape (3, 480)  [nasal / spo2 / thoracic]
  ↓
Train BreathingCNN with Focal Loss
  ↓
Leave-One-Participant-Out Cross Validation (5 folds)
  ↓
Results + Confusion Matrix + Per-fold chart
```

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────
import torch

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — go to Settings → Accelerator → GPU T4 x1')

In [ ]:
# ── Cell 2: Imports ───────────────────────────
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {DEVICE}')

In [ ]:
from pathlib import Path

# Auto-detect where combined_dataset.csv lives under /kaggle/input
# Works for both path formats Kaggle uses
def find_dataset_dir(filename):
    for p in Path('/kaggle/input').rglob(filename):
        return p.parent
    return None

INPUT_DIR  = find_dataset_dir('combined_dataset.csv')
OUTPUT_DIR = Path('/kaggle/working')

assert INPUT_DIR is not None, (
    'combined_dataset.csv not found under /kaggle/input\n'
    'Attach your dataset in the right panel (Add Input).'
)

CSV_PATH = INPUT_DIR / 'combined_dataset.csv'
# cnn_model.py is written inline in Cell 4 (no upload needed)
MODEL_PY = OUTPUT_DIR / 'cnn_model.py'

FS           = 32
WIN_SEC      = 15
STEP_SEC     = 7
OV_THR       = 0.4
EPOCHS       = 50
BATCH_SIZE   = 128
LR           = 5e-4
DROPOUT      = 0.4

WIN_SAMPLES  = WIN_SEC  * FS
STEP_SAMPLES = STEP_SEC * FS

print(f'Dataset  : {INPUT_DIR}')
print(f'Window   : {WIN_SEC}s = {WIN_SAMPLES} samples')
print(f'Step     : {STEP_SEC}s = {STEP_SAMPLES} samples')
print(f'Epochs   : {EPOCHS}   |   Batch: {BATCH_SIZE}   |   LR: {LR}')
csv_ok = 'OK' if CSV_PATH.exists() else 'NOT FOUND'
print(f'CSV      : {csv_ok}')


In [ ]:
# ── Cell 4: Define and import BreathingCNN ──
import sys, torch, torch.nn as nn

# Write cnn_model.py to /kaggle/working at runtime
# (no need to upload it — only combined_dataset.csv needed)
MODEL_CODE = (
    'import torch\n'
    'import torch.nn as nn\n'
    '\n'
    'class ConvBlock(nn.Module):\n'
    '    def __init__(self, in_ch, out_ch, kernel, pool=2):\n'
    '        super().__init__()\n'
    '        self.block = nn.Sequential(\n'
    '            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=kernel//2),\n'
    '            nn.BatchNorm1d(out_ch),\n'
    '            nn.ReLU(),\n'
    '            nn.MaxPool1d(pool),\n'
    '        )\n'
    '    def forward(self, x):\n'
    '        return self.block(x)\n'
    '\n'
    'class BreathingCNN(nn.Module):\n'
    '    def __init__(self, n_classes, window_samples, dropout=0.4):\n'
    '        super().__init__()\n'
    '        self.block_a = ConvBlock(3, 32, kernel=7)\n'
    '        self.block_b = ConvBlock(32, 64, kernel=5)\n'
    '        self.block_c = nn.Sequential(\n'
    '            nn.Conv1d(64, 128, kernel_size=3, padding=1),\n'
    '            nn.BatchNorm1d(128),\n'
    '            nn.ReLU(),\n'
    '            nn.AdaptiveAvgPool1d(1),\n'
    '        )\n'
    '        self.classifier = nn.Sequential(\n'
    '            nn.Flatten(),\n'
    '            nn.Linear(128, 64),\n'
    '            nn.ReLU(),\n'
    '            nn.Dropout(dropout),\n'
    '            nn.Linear(64, n_classes),\n'
    '        )\n'
    '    def forward(self, x):\n'
    '        x = self.block_a(x)\n'
    '        x = self.block_b(x)\n'
    '        x = self.block_c(x)\n'
    '        return self.classifier(x)\n'
    '    def count_parameters(self):\n'
    '        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n'
)

MODEL_PY.write_text(MODEL_CODE)
print(f'Wrote {MODEL_PY}  ({MODEL_PY.stat().st_size} bytes)')

# Import from /kaggle/working
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

# Force reload in case a stale version is cached
import importlib
if 'cnn_model' in sys.modules:
    del sys.modules['cnn_model']

from cnn_model import BreathingCNN

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Shape trace
_m = BreathingCNN(n_classes=4, window_samples=WIN_SAMPLES).to(DEVICE)
_x = torch.randn(4, 3, WIN_SAMPLES).to(DEVICE)
with torch.no_grad():
    _a = _m.block_a(_x)
    _b = _m.block_b(_a)
    _c = _m.block_c(_b)
    _o = _m(_x)
print('BreathingCNN shape trace:')
print(f'  Input   : {tuple(_x.shape)}')
print(f'  Block A : {tuple(_a.shape)}  (32 filters, time/2)')
print(f'  Block B : {tuple(_b.shape)}   (64 filters, time/4)')
print(f'  Block C : {tuple(_c.shape)}  (128 filters, collapsed)')
print(f'  Output  : {tuple(_o.shape)}')
print(f'  Params  : {_m.count_parameters():,}')
del _m, _x, _a, _b, _c, _o


In [ ]:
# ── Cell 5: Load dataset ──────────────────────

df = pd.read_csv(CSV_PATH, parse_dates=['datetime'])
df = df.sort_values(['participant', 'datetime']).reset_index(drop=True)

# Merge rare classes into Obstructive Apnea
# Body event + Mixed Apnea are physiologically similar to OSA
# and have too few samples to learn from separately
df['event_label'] = df['event_label'].replace({
    'Body event':  'Obstructive Apnea',
    'Mixed Apnea': 'Obstructive Apnea',
})

print(f'Total rows : {len(df):,}')
print(f'Columns    : {list(df.columns)}')
print()
print('Event distribution after merging:')
for lbl, cnt in df['event_label'].value_counts().items():
    pct = cnt / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {lbl:22s}  {cnt:>8,}  ({pct:5.1f}%)  {bar}')

In [ ]:
# ── Cell 6: Build windows ─────────────────────
#
# A "window" is a fixed-length slice of the signal.
# We slide it along the recording with STEP_SAMPLES overlap.
#
# Each window becomes one training sample:
#   X shape : (3, WIN_SAMPLES)  — 3 signal channels
#   y       : string label      — majority event in that window
#
# Labelling rule:
#   If ≥ OV_THR (40%) of the window samples are labelled with
#   an event, the window gets that event label.
#   Otherwise → Normal.

def get_window_label(labels, thr=OV_THR):
    counts = Counter(labels)
    n      = len(labels)
    # Ignore Normal when looking for event labels
    events = {k: v for k, v in counts.items() if k != 'Normal'}
    if not events:
        return 'Normal'
    best_label, best_count = max(events.items(), key=lambda kv: kv[1])
    return best_label if best_count / n >= thr else 'Normal'


def build_windows(df_pid):
    nasal    = df_pid['nasal'].values.astype(np.float32)
    spo2     = df_pid['spo2'].values.astype(np.float32)
    thoracic = df_pid['thoracic'].values.astype(np.float32)
    labels   = df_pid['event_label'].values
    n        = len(nasal)

    X_windows, y_labels = [], []
    for start in range(0, n - WIN_SAMPLES + 1, STEP_SAMPLES):
        end = start + WIN_SAMPLES
        # Stack 3 channels into shape (3, WIN_SAMPLES)
        window = np.stack([
            nasal[start:end],
            spo2[start:end],
            thoracic[start:end],
        ])
        X_windows.append(window)
        y_labels.append(get_window_label(labels[start:end]))

    return np.array(X_windows, dtype=np.float32), np.array(y_labels)


print('Building windows per participant...')
PIDS        = sorted(df['participant'].unique())
pid_windows = {}

for pid in PIDS:
    X, y  = build_windows(df[df['participant'] == pid].reset_index(drop=True))
    pid_windows[pid] = (X, y)
    counts   = Counter(y)
    n_event  = sum(v for k, v in counts.items() if k != 'Normal')
    print(f'  {pid}: {len(X):>6,} windows  |  '
          f'Normal={counts["Normal"]:,}  '
          f'Events={n_event:,} ({n_event/len(X)*100:.1f}%)')

# Fit label encoder on all labels across all participants
all_labels = np.concatenate([y for _, y in pid_windows.values()])
le         = LabelEncoder().fit(all_labels)
N_CLASSES  = len(le.classes_)

print(f'\nClasses ({N_CLASSES}): {list(le.classes_)}')
print(f'Total windows : {len(all_labels):,}')

In [ ]:
# ── Cell 7: Dataset class + helpers ──────────

class WindowDataset(Dataset):
    """Wraps numpy arrays as a PyTorch Dataset."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


def get_norm_stats(X: np.ndarray):
    """
    Compute per-channel mean and std across all windows and time steps.
    Shape of X: (N, 3, T)
    Returns mean, std each shape (1, 3, 1) — broadcast-ready.

    Why normalise?
    Nasal values are ~100s, SpO2 is ~90-100, thoracic is ~10s.
    Without normalisation the network is dominated by the largest signal.
    """
    mean = X.mean(axis=(0, 2), keepdims=True)
    std  = X.std(axis=(0, 2),  keepdims=True) + 1e-8
    return mean, std

def normalise(X, mean, std):
    return (X - mean) / std


print('✅ WindowDataset, get_norm_stats, normalise — ready')

In [ ]:
# ── Cell 8: Focal Loss ────────────────────────
#
# Normal windows are ~87% of data.
# Standard CrossEntropyLoss: the model learns to predict Normal
# for everything and achieves 87% accuracy while missing all apneas.
#
# Focal Loss solution:
#   loss = (1 - p_correct)^gamma  ×  CrossEntropy
#
#   p_correct = probability the model assigned to the TRUE class.
#
#   If model is CONFIDENT and CORRECT → p_correct ≈ 1.0
#     → (1 - 1)^2 = 0  → this sample barely contributes to the gradient
#     → easy Normal windows are ignored
#
#   If model is WRONG or UNSURE → p_correct ≈ 0.1
#     → (1 - 0.1)^2 = 0.81  → full gradient signal
#     → hard event windows drive the learning
#
#   gamma=2 is the standard value from the original paper (Lin et al. 2017)

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, reduction='none')
        p_t   = torch.exp(-ce)                 # prob of true class
        focal = (1 - p_t) ** self.gamma * ce   # down-weight easy samples
        return focal.mean()


print('✅ FocalLoss (gamma=2) — ready')
print('   Easy Normal windows → near-zero loss contribution')
print('   Hard event windows  → full loss contribution')

In [ ]:
# ── Cell 9: Train + eval helpers ─────────────

def train_one_epoch(model, loader, optimiser, criterion):
    model.train()
    total_loss, correct, n = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimiser.zero_grad()
        logits = model(X_batch)               # forward pass
        loss   = criterion(logits, y_batch)   # focal loss
        loss.backward()                        # backprop

        # Gradient clipping: prevents any one batch from causing
        # a huge weight update that destabilises training
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()

        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        n          += len(y_batch)

    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader):
    """Returns (predictions, true_labels) as numpy arrays."""
    model.eval()
    preds_all, tgts_all = [], []

    for X_batch, y_batch in loader:
        logits = model(X_batch.to(DEVICE))
        preds_all.extend(logits.argmax(1).cpu().numpy())
        tgts_all.extend(y_batch.numpy())

    return np.array(preds_all), np.array(tgts_all)


print('✅ train_one_epoch, evaluate — ready')

In [ ]:
# ── Cell 10: LOPO-CV Training Loop ───────────
#
# LOPO = Leave One Participant Out
#
# Why not random train/test split?
# A random split would mix the same person's windows into both
# train and test. The model would memorise that person's breathing
# style and get artificially high accuracy.
#
# LOPO tests on a completely unseen person each fold — closer
# to how the model would work in the real world.
#
# 5 folds: each participant gets to be the test set once.

all_label_ids = list(range(N_CLASSES))
lopo_preds    = []
lopo_tgts     = []
fold_rows     = []

for fold_idx, test_pid in enumerate(PIDS, start=1):

    train_pids = [p for p in PIDS if p != test_pid]

    print(f'\n{"═" * 60}')
    print(f'  Fold {fold_idx}/{len(PIDS)}   '
          f'Test: {test_pid}   Train: {train_pids}')
    print(f'{"═" * 60}')

    # ── Assemble train and test arrays ─────────────────────
    X_tr = np.concatenate([pid_windows[p][0] for p in train_pids])
    y_tr = np.concatenate([pid_windows[p][1] for p in train_pids])
    X_te = pid_windows[test_pid][0]
    y_te = pid_windows[test_pid][1]

    # ── Normalise (ALWAYS fit on train only — no leakage) ──
    norm_mean, norm_std = get_norm_stats(X_tr)
    X_tr = normalise(X_tr, norm_mean, norm_std)
    X_te = normalise(X_te, norm_mean, norm_std)

    # ── Encode string labels → integers ───────────────────
    y_tr_enc = le.transform(y_tr)
    y_te_enc = le.transform(y_te)

    print(f'  Train: {len(X_tr):,} windows   Test: {len(X_te):,} windows')
    print(f'  Train label breakdown:')
    for lbl, cnt in Counter(y_tr).most_common():
        print(f'    {lbl:25s} {cnt:>6,}  ({cnt/len(y_tr)*100:.1f}%)')

    # ── DataLoaders ────────────────────────────────────────
    train_loader = DataLoader(
        WindowDataset(X_tr, y_tr_enc),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        WindowDataset(X_te, y_te_enc),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )

    # ── Build model ────────────────────────────────────────
    model = BreathingCNN(
        n_classes=N_CLASSES,
        window_samples=WIN_SAMPLES,
        dropout=DROPOUT
    ).to(DEVICE)
    print(f'\n  Model params: {model.count_parameters():,}')

    # ── Loss, optimiser, scheduler ─────────────────────────
    criterion = FocalLoss(gamma=2.0)

    # AdamW = Adam + proper weight decay (decoupled from gradient)
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=1e-3
    )

    # CosineAnnealingLR: smoothly reduces LR from LR → LR/20
    # Avoids sudden LR drops that cause training instability
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=EPOCHS, eta_min=LR / 20
    )

    # ── Training loop ──────────────────────────────────────
    print()
    for epoch in range(1, EPOCHS + 1):
        loss, acc = train_one_epoch(model, train_loader, optimiser, criterion)
        scheduler.step()
        if epoch % 10 == 0 or epoch == 1:
            lr_now = scheduler.get_last_lr()[0]
            print(f'  Epoch {epoch:2d}/{EPOCHS}  '
                  f'loss={loss:.4f}  acc={acc:.4f}  lr={lr_now:.2e}')

    # ── Evaluate ───────────────────────────────────────────
    fold_preds, fold_tgts = evaluate(model, test_loader)
    fold_acc = accuracy_score(fold_tgts, fold_preds)

    print(f'\n  ── Fold {fold_idx} Results ({test_pid}) ──')
    print(f'  Accuracy : {fold_acc:.4f}')
    print()
    print(classification_report(
        fold_tgts, fold_preds,
        labels=all_label_ids,
        target_names=le.classes_,
        zero_division=0
    ))

    cm = confusion_matrix(fold_tgts, fold_preds, labels=all_label_ids)
    cm_df = pd.DataFrame(
        cm,
        index=[f'T:{c}' for c in le.classes_],
        columns=[f'P:{c}' for c in le.classes_]
    )
    print(cm_df.to_string())

    # ── Save checkpoint ────────────────────────────────────
    # Saves model weights + normalisation stats together
    # so the model can be loaded and used without recomputing stats
    ckpt = {
        'model_state': model.state_dict(),
        'classes':     list(le.classes_),
        'norm_mean':   norm_mean.tolist(),
        'norm_std':    norm_std.tolist(),
        'win_samples': WIN_SAMPLES,
    }
    ckpt_path = OUTPUT_DIR / f'cnn_fold{fold_idx}_{test_pid}.pt'
    torch.save(ckpt, ckpt_path)
    print(f'\n  ✅ Saved → {ckpt_path.name}')

    # Accumulate for overall report
    lopo_preds.extend(fold_preds.tolist())
    lopo_tgts.extend(fold_tgts.tolist())
    fold_rows.append({
        'fold':     fold_idx,
        'test_pid': test_pid,
        'n_test':   len(fold_tgts),
        'accuracy': round(fold_acc, 4),
    })

lopo_preds = np.array(lopo_preds)
lopo_tgts  = np.array(lopo_tgts)
print('\n✅ All 5 folds complete')

In [ ]:
# ── Cell 11: Overall results ──────────────────

fold_df  = pd.DataFrame(fold_rows)
ov_acc   = accuracy_score(lopo_tgts, lopo_preds)

print('\n' + '═' * 60)
print('  OVERALL LOPO-CV RESULTS')
print('═' * 60)
print(f'  Overall Accuracy : {ov_acc:.4f}  ({ov_acc*100:.2f}%)')
print()
print(classification_report(
    lopo_tgts, lopo_preds,
    labels=all_label_ids,
    target_names=le.classes_,
    zero_division=0
))

print('Per-fold summary:')
print(fold_df.to_string(index=False))
print(f'\nMean accuracy : {fold_df["accuracy"].mean():.4f} '
      f'± {fold_df["accuracy"].std():.4f}')

fold_df.to_csv(OUTPUT_DIR / 'lopo_results_cnn.csv', index=False)
print(f'\n✅ Saved → lopo_results_cnn.csv')

In [ ]:
# ── Cell 12: Plots ────────────────────────────

# --- Confusion matrix ---
ov_cm = confusion_matrix(lopo_tgts, lopo_preds, labels=all_label_ids)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=ov_cm,
    display_labels=le.classes_
).plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title(
    'Overall Confusion Matrix — LOPO-CV\n'
    'BreathingCNN  |  Focal Loss  |  5 participants',
    fontsize=11, fontweight='bold'
)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_cnn.png',
            dpi=150, bbox_inches='tight')
plt.show()

# --- Per-fold accuracy bar chart ---
fig, ax = plt.subplots(figsize=(8, 4))
x    = np.arange(len(fold_df))
bars = ax.bar(x, fold_df['accuracy'], color='#1565C0', width=0.5)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2,
            h + 0.008, f'{h:.3f}',
            ha='center', va='bottom', fontsize=9)

mean_acc = fold_df['accuracy'].mean()
ax.axhline(mean_acc, color='red', linestyle='--', lw=1.5,
           label=f'Mean = {mean_acc:.3f}')
ax.set_xticks(x)
ax.set_xticklabels(
    [f"Fold {r['fold']}\n({r['test_pid']})" for _, r in fold_df.iterrows()]
)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Test Accuracy')
ax.set_title('Per-Fold Test Accuracy — LOPO-CV', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_fold_accuracy_cnn.png',
            dpi=150, bbox_inches='tight')
plt.show()

# --- Summary ---
print('Output files saved to /kaggle/working:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1e3
    print(f'  {f.name:40s}  {size_kb:>7.0f} KB')